# 1. Quickstart: your first SVD check

## What this notebook is about

You have, or want to write, a function that computes a singular value decomposition (SVD),
`A = U · diag(s) · Vᴴ`. Before anyone can use it, you need to know three things: **is it
correct** on hard matrices too, **how fast** is it, and **does it really run on the GPU**.
`svdlab` answers those with one call. You give it the function. It runs the function on a
fixed set of test matrices, compares each result with a trusted reference SVD, and reports
a verdict, the errors and the time for each matrix.

In this notebook you don't write any code yet. You run a working SVD that ships with the
repository, read its report, compare it with other methods, and see the same thing as a
terminal command. It takes a few minutes on a laptop CPU.

## What you will learn

1. The *contract*: what a function must take and return so `svdlab` can check it.
2. How to run the check and read every field of a report line.
3. How to compare several methods on the same matrices.
4. The terminal command that does the same, and its exit code.
5. (With a GPU) how to confirm that a method works on the GPU.

## Words used here

| Word | Meaning |
| --- | --- |
| **method** | anything that computes an SVD: a Python function, a file with a function `svd`, or a builtin name such as `'torch'` |
| **case** | one test matrix plus the number of singular triplets asked for |
| **suite** | a fixed list of cases: `QUICK_SUITE` (6 small matrices, for a first try) or `DEFAULT_SUITE` (8 larger ones, for reporting) |
| **reference** | an SVD of the same matrix computed with LAPACK in float64; every result is compared with it |
| **verdict** | `ok`, `FAIL` (wrong result) or `ERROR` (the method crashed) |

More words and a map of the repository: [Start here](../docs/tutorials/00-start-here.md).

## Before you start

Install the package once, from the repository root, in the Python environment this
notebook uses:

```bash
python -m pip install -e '.[laboratory]'
```

([Tutorial 1](../docs/tutorials/01-install-and-first-check.md) explains how to install
PyTorch for your GPU first.) Then run the cells from top to bottom: Shift+Enter runs one
cell, *Run → Run All Cells* runs everything. Read the text between the cells as you go.

## Setup

**What this cell does:** it finds the repository folder (so the notebook works from any
working directory), makes `svdlab` importable even if you have not installed it with pip,
and chooses where to compute: `DEVICE` is `'cuda'` when PyTorch sees a GPU and `'cpu'`
otherwise. To stay on the CPU even with a GPU present, set `USE_GPU = False`.

**What you should see:** one line with the PyTorch version and the chosen device. If you
get `ModuleNotFoundError`, the notebook kernel is not the Python environment where you ran
`pip install -e '.[laboratory]'`. Choose that environment as the kernel and run the cell
again.

In [1]:
# Setup: find the repository, choose the device. Run this cell first.
from pathlib import Path
import os, sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'svdlab' / '__init__.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))      # works even if the package is not installed with pip
os.chdir(ROOT)                         # all paths below are relative to the repository root

import torch
import svdlab

USE_GPU = True                         # set to False to stay on the CPU even when a GPU is present
DEVICE = 'cuda' if USE_GPU and torch.cuda.is_available() else 'cpu'
torch.set_num_threads(4)               # same CPU thread count as the command line, comparable timings
print(f'torch {torch.__version__}, device: {DEVICE}' + (f' ({torch.cuda.get_device_name(0)})' if DEVICE == 'cuda' else ''))

torch 2.14.1+cu130, device: cpu


## Step 1. The contract: what a method looks like

Every method is a function with this signature:

```python
def svd(a, rank):          # a: torch tensor (m, n), real or complex; rank <= min(m, n)
    ...
    return U, s, Vh        # (m, k), (k,), (k, n) with k >= rank, s sorted from largest to smallest
```

- `a` is the test matrix, already on `DEVICE`.
- `rank` is how many singular triplets this case needs: `min(m, n)` for a full SVD, fewer
  for a low-rank case. Returning more is fine; the extra ones are cut off.
- `U`, `s`, `Vh` are the factors, so that `a ≈ U @ diag(s) @ Vh`.

The repository ships a working method to start from: `examples/my_svd.py`, a *one-sided
Jacobi* SVD written in plain PyTorch. (Jacobi rotates pairs of columns of A until they are
all orthogonal. It is slow but very accurate. You don't need the details now.)

**What this cell does:** `svdlab.load_file` loads the function `svd` from that file, the same
way the command line does. Then it calls the function on one random 6×4 matrix, as a
quick smoke test, before the real check.

**What you should see:** the shapes `(6, 4) (4,) (4, 4)`, four singular values sorted from
largest to smallest, and a reconstruction error around 1e-15, which is the precision of
float64.

In [2]:
svd = svdlab.load_file('examples/my_svd.py')

a = torch.randn(6, 4, dtype=torch.float64)
u, s, vh = svd(a, 4)
print('shapes:', tuple(u.shape), tuple(s.shape), tuple(vh.shape))
print('singular values:', s)
print('|A - U diag(s) Vh| =', float(torch.linalg.norm(a - u * s @ vh)))   # u * s scales column j by s[j]

shapes: (6, 4) (4,) (4, 4)
singular values: tensor([3.9409, 3.0601, 2.3853, 1.2552], dtype=torch.float64)
|A - U diag(s) Vh| = 3.0559585155205097e-15


## Step 2. Run the check

**What this cell does:** `svdlab.check` runs the method on every case of the suite. For
each case it:

1. creates the matrix from a fixed seed (the same numbers on every machine);
2. computes the float64 reference SVD;
3. calls your function, compares the result with the reference and computes the errors;
4. calls it again a few times and measures the median time of the call alone.

We use `QUICK_SUITE` (6 small matrices) so it finishes in seconds.

**What you should see:** one line per matrix, all `ok`. The first lines may take a moment
while PyTorch warms up.

In [3]:
rows = svdlab.check(svd, svdlab.QUICK_SUITE, name='my_svd', device=DEVICE)

my_svd               gaussian_tall          ok      65.981 ms  recon 4.51e-15  tail× -  σ-rel 3.64e-15  orth 4.66e-15


my_svd               gaussian_wide          ok      48.535 ms  recon 4.34e-15  tail× -  σ-rel 3.71e-15  orth 4.25e-15


my_svd               ill_conditioned_1e8    ok     123.728 ms  recon 6.48e-15  tail× -  σ-rel 4.24e-10  orth 6.60e-15


my_svd               clustered              ok      83.124 ms  recon 8.67e-15  tail× -  σ-rel 4.75e-14  orth 8.29e-15


my_svd               complex_gaussian       ok      34.863 ms  recon 3.04e-15  tail× -  σ-rel 2.44e-15  orth 3.20e-15


my_svd               low_rank_k4            ok     347.271 ms  recon 6.64e-06  tail× 1.00e+00  σ-rel 3.99e-15  orth 7.96e-15


### How to read a line

Take the first line as an example:

```text
my_svd   gaussian_tall   ok   64.858 ms   recon 4.51e-15   tail× -   σ-rel 3.64e-15   orth 4.66e-15
```

| Field | Meaning |
| --- | --- |
| method name | the function or file being checked |
| case name | the test matrix, for example `gaussian_tall` (48×32 random Gaussian, float64) |
| `ok` / `FAIL` / `ERROR` | the verdict: correct within tolerance / computed but wrong / the method raised an exception |
| `ms` | median time of one call of your function; nothing else is timed |
| `recon` | ‖A − U diag(s) Vᴴ‖ / ‖A‖: how well the factors rebuild the matrix |
| `tail×` | truncated cases only: your residual divided by the best possible rank-k residual (1.00 is optimal) |
| `σ-rel` | largest relative error of a singular value |
| `orth` | how far U and V are from having orthonormal columns |

Errors around 1e-15 are as good as float64 gets. Two lines deserve a second look:

- **`ill_conditioned_1e8`** shows `σ-rel` around 1e-10, much larger than the others, and is
  still `ok`. This matrix has singular values from 1 down to 1e-8. The tiny ones cannot be
  computed to 15 correct digits by any float64 method, because the rounding errors in the
  input already change them in the 10th digit. The verdict therefore measures errors
  relative to the whole matrix (`recon`, `orth`), and those are still around 1e-15.
- **`low_rank_k4`** asks for only 4 triplets of an 80×60 matrix. `recon` is 6.6e-6 because
  the part of the matrix that was left out is that large. `tail× 1.00` says this is exactly
  as good as the best possible rank-4 approximation, so the case is `ok`.

### All the numbers as a table

**What this cell does:** `check` returned one Python dictionary per case (we call them
*rows*). `svdlab.table` turns them into a pandas DataFrame, which is easier to read and to
sort. Here we show a few of the about twenty columns, with their full names.
[Notebook 5](05_metrics_and_matrices.ipynb) explains all of them.

In [4]:
svdlab.table(rows)[['case', 'm', 'n', 'dtype', 'rank', 'ok', 'time_median_ms',
                    'reconstruction_relative_fro', 'resolved_spectrum_max_relative_error',
                    'orthogonality_u_fro_over_sqrt_cols']]

,case,m,n,dtype,rank,ok,time_median_ms,reconstruction_relative_fro,resolved_spectrum_max_relative_error,orthogonality_u_fro_over_sqrt_cols
0,gaussian_tall,48,32,float64,32,True,65.981010,4.507216e-15,3.641159e-15,1.545500e-15
1,gaussian_wide,24,40,float64,24,True,48.534984,4.339813e-15,3.708041e-15,4.246111e-15
2,ill_conditioned_1e8,40,30,float64,30,True,123.728047,6.480045e-15,4.239494e-10,2.221803e-15
3,clustered,40,30,float64,30,True,83.124469,8.666571e-15,4.748805e-14,1.903608e-15
4,complex_gaussian,32,20,complex128,20,True,34.862677,3.042918e-15,2.440787e-15,1.342602e-15
5,low_rank_k4,80,60,float64,4,True,347.271107,6.635512e-06,3.993729e-15,4.910392e-16


## Step 3. Compare with other methods

A result means more next to others. `svdlab.compare` runs several methods on the same
matrices. A method can be a function, a file name, or a builtin name:

- `'torch'` is `torch.linalg.svd`: LAPACK on the CPU, cuSOLVER on the GPU. It is the
  speed and accuracy baseline.
- `'jacobi'` is the repository's own Jacobi SVD.
- `python -m svdlab list` prints all builtin names (16 repository SVDs plus a few more).

**What this cell does:** it runs three methods on the quick suite without printing every
line (`verbose=False`), then prints one summary line per method with `svdlab.summary`.

**What you should see:** all three 6/6 ok. `torch` is hundreds of times faster: it is
compiled vendor code, while `my_svd` and `jacobi` are Python loops around tensor
operations. That is normal for a readable prototype. The point is to get it correct
first, then fast.

In [5]:
rows = svdlab.compare({'my_svd': svd, 'torch': 'torch', 'jacobi': 'jacobi'}, svdlab.QUICK_SUITE,
                      device=DEVICE, verbose=False)
for s in svdlab.summary(rows):
    print(f"{s['method']:<8} {s['ok']}/{s['cases']} ok   worst recon {s['worst_reconstruction']:.1e}   "
          f"total {s['total_median_ms']:8.2f} ms")

my_svd   6/6 ok   worst recon 6.6e-06   total   811.14 ms
torch    6/6 ok   worst recon 6.6e-06   total     1.23 ms
jacobi   6/6 ok   worst recon 6.6e-06   total   779.91 ms


**What this cell does:** it shows the median time in milliseconds per case and method,
using a pandas pivot table: one row per case, one column per method.

In [6]:
svdlab.table(rows).pivot(index='case', columns='method', values='time_median_ms').round(3)

method,jacobi,my_svd,torch
case,,,
clustered,154.063,70.018,0.161
complex_gaussian,64.480,52.678,0.259
gaussian_tall,80.246,120.372,0.214
gaussian_wide,79.375,77.811,0.150
ill_conditioned_1e8,109.309,167.420,0.113
low_rank_k4,292.436,322.838,0.335


## Step 4. The same from the terminal

Everything above is a single terminal command, run from the repository root:

```bash
python -m svdlab check examples/my_svd.py --quick --compare torch,jacobi
```

- `check examples/my_svd.py`: the method to check (a file with a function `svd`).
- `--quick`: use the 6 small matrices instead of the 8 standard ones.
- `--compare torch,jacobi`: also run these for reference. They never change the exit code.

The exit code is 0 when every case of the checked method is `ok`, 1 when one fails and 2
when the method cannot be loaded. So the command also works in scripts and CI.

**What this cell does:** it defines a small helper, `svdlab_cli('...')`, that runs
`python -m svdlab ...` inside the notebook. That way you see exactly what the terminal
command prints, without leaving the notebook. In a terminal you would type the same words
after `python -m svdlab`.

In [7]:
import shlex
from svdlab.__main__ import main as svdlab_main

def svdlab_cli(command):
    """Run `python -m svdlab <command>` inside this notebook (same output, same exit code)."""
    code = svdlab_main(shlex.split(command))
    print(f'exit code: {code}')

**What this cell does:** it runs the command above. **What you should see:** a short legend,
one line per method and case, a summary per method, and `exit code: 0`.

In [8]:
svdlab_cli('check examples/my_svd.py --quick --compare torch,jacobi')

recon  = |A - U diag(s) Vh| / |A|          tail× = recon / best possible (truncated cases)
σ-rel  = max relative error of the values   orth  = max(|UᴴU - I|, |V Vᴴ - I|) / sqrt(k)
ok     = orth and recon within 100·eps·max(m, n), or tail× <= 1.05



my_svd               gaussian_tall          ok     104.540 ms  recon 4.51e-15  tail× -  σ-rel 3.64e-15  orth 4.66e-15


my_svd               gaussian_wide          ok      63.842 ms  recon 4.34e-15  tail× -  σ-rel 3.71e-15  orth 4.25e-15


my_svd               ill_conditioned_1e8    ok     144.065 ms  recon 6.48e-15  tail× -  σ-rel 4.24e-10  orth 6.60e-15


my_svd               clustered              ok      75.912 ms  recon 8.67e-15  tail× -  σ-rel 4.75e-14  orth 8.29e-15


...

jacobi               gaussian_wide          ok      40.042 ms  recon 5.65e-15  tail× -  σ-rel 4.71e-15  orth 5.24e-15


jacobi               ill_conditioned_1e8    ok     116.582 ms  recon 7.45e-15  tail× -  σ-rel 1.51e-09  orth 8.77e-15


jacobi               clustered              ok     114.521 ms  recon 1.02e-14  tail× -  σ-rel 1.99e-14  orth 9.73e-15


jacobi               complex_gaussian       ok      34.563 ms  recon 3.55e-15  tail× -  σ-rel 2.95e-15  orth 3.63e-15


jacobi               low_rank_k4            ok     284.713 ms  recon 6.64e-06  tail× 1.00e+00  σ-rel 4.64e-15  orth 9.03e-15

my_svd: 6/6 ok, total median 662.740 ms
torch: 6/6 ok, total median 1.424 ms
jacobi: 6/6 ok, total median 655.627 ms
exit code: 0


## Step 5. On the GPU (optional)

Running on a GPU can go wrong in two quiet ways: the code may copy the data back to the
CPU and compute there (correct numbers, meaningless GPU timing), or the GPU path may give
different numbers from the CPU path. `svdlab.gpu_check` catches both. It runs the method on
the CPU and on the GPU, and a case passes only when:

- both runs are `ok`,
- the GPU result is really on the GPU (`output_device`),
- the singular values from both runs agree within the tolerance.

**What this cell does:** on a machine with a GPU, it runs that check and shows one row per
case. On a CPU-only machine it does nothing and says so. Terminal equivalent:
`python -m svdlab check examples/my_svd.py --quick --gpu-check`.

**What you should see (GPU):** `passed` True for every case, `output_device` `cuda:0`, and
the speedup of the GPU over the CPU. For matrices this small the GPU is often not faster:
launching GPU work has a fixed cost that large matrices amortize.

In [9]:
if DEVICE == 'cuda':
    gpu_rows = svdlab.gpu_check(svd, svdlab.QUICK_SUITE, verbose=False)
    display(svdlab.table(gpu_rows)[['case', 'passed', 'cpu_ok', 'gpu_ok', 'output_device', 'speedup']])
else:
    print('No CUDA/ROCm device visible to torch: GPU check skipped.')

No CUDA/ROCm device visible to torch: GPU check skipped.


## Summary

- A method is a function `svd(a, rank) -> U, s, Vh`.
- `svdlab.check(method, suite)` or `python -m svdlab check file.py` runs it on fixed test
  matrices and gives a verdict, errors and time per matrix.
- `compare` puts several methods side by side; `gpu_check` validates a GPU run.
- Large `σ-rel` on an ill-conditioned matrix, or large `recon` on a truncated case, can be
  perfectly correct. The verdict accounts for that.

## Next

- [02_write_your_own_svd.ipynb](02_write_your_own_svd.ipynb): write an SVD, see it fail, fix it.
- [03_blocks_and_recipes.ipynb](03_blocks_and_recipes.ipynb): replace one step of an existing SVD.
- [04_native_cpp_cuda.ipynb](04_native_cpp_cuda.ipynb): C++ and CUDA.
- [05_metrics_and_matrices.ipynb](05_metrics_and_matrices.ipynb): every metric, your own matrices.